# Meeting Accountability Detector - Data & Baseline

Turn the AMI meetings into labeled utterances, split them, and run a simple
baseline. An utterance is **positive** if it was marked as an action item in the
meeting summary. This gives 381 action items, the same number used in past work.


## 1. Setup

Install the libraries and point at the data folder.

In [ ]:
# %pip install -q pandas scikit-learn pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import re
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd, numpy as np

AMI = Path("data/ami_public_manual_1.6.2")   # the extracted AMI folder
assert (AMI/"words").is_dir() and (AMI/"extractive").is_dir(), \
    "Can't find data/ami_public_manual_1.6.2/. Extract the AMI zip there first."

## 2. Read the data and add labels

The AMI files are XML. We read the words, group them into utterances (dialogue
acts), and mark an utterance positive if the meeting summary linked it to an
**action**. We only keep meetings that have a summary.

In [32]:
NITE = "{http://nite.sourceforge.net/}"
def nid(el):   return el.get(NITE + "id")                    # the nite:id value
def ids(href): return re.findall(r"id\(([^)]+)\)", href)     # ids inside #id(...)

def word_range(href):
    "Turn id(a)..id(b) into every word id from a to b."
    g = ids(href)
    if len(g) < 2: return g
    a, b = g
    na, nb = re.search(r"\d+$", a), re.search(r"\d+$", b)
    return [a[:na.start()] + str(i) for i in range(int(na.group()), int(nb.group())+1)] if na and nb else g

def join(tokens):
    "Glue word tokens into a sentence, keeping punctuation tidy."
    out = ""
    for t in tokens:
        if not t: continue
        out += t if (not out or re.match(r"[.,!?;:'\")]", t)) else " " + t
    return out.strip()

def read_words(path):
    if not path.exists(): return {}
    return {nid(w): (w.text or "").strip() for w in ET.parse(path).getroot() if w.tag == "w"}

def read_dialogue_acts(path, words):
    "Yield one utterance per dialogue act, with its text and type."
    if not path.exists(): return
    for dact in ET.parse(path).getroot():
        if dact.tag != "dact": continue
        wids, da_type = [], None
        for ch in dact:
            href = ch.get("href", "")
            if   ch.tag == NITE+"child"   and "words" in href:     wids += word_range(href)
            elif ch.tag == NITE+"pointer" and "da-types" in href:  da_type = (ids(href) or [None])[0]
        toks = [words.get(w, "") for w in wids]
        yield dict(da_id=nid(dact), da_type=da_type, text=join(toks),
                   n_words=sum(1 for t in toks if t))

def action_sentences(path):
    "Ids of summary sentences under the ACTIONS heading."
    if not path.exists(): return set()
    return {nid(s) for sec in ET.parse(path).getroot() if sec.tag == "actions"
            for s in sec if s.tag == "sentence"}

def action_dialogue_acts(path, action_ids):
    "Dialogue acts the summary linked to an ACTIONS sentence = the positives."
    if not path.exists(): return set()
    hits = set()
    for link in ET.parse(path).getroot():
        if link.tag != "summlink": continue
        da = ab = None
        for ptr in link:
            href = ptr.get("href", "")
            if   "dialog-act" in href: da = (ids(href) or [None])[0]
            elif "abssumm"    in href: ab = (ids(href) or [None])[0]
        if ab in action_ids: hits.add(da)
    return hits

In [33]:
rows = []
parts = sorted({p.name.split(".")[0] for p in (AMI/"words").glob("*.words.xml")})

for part in parts:
    abss = AMI/"abstractive"/f"{part}.abssumm.xml"
    summ = AMI/"extractive"/f"{part}.summlink.xml"
    if not (abss.exists() and summ.exists()):     # skip meetings without a summary
        continue
    positives = action_dialogue_acts(summ, action_sentences(abss))
    meeting = re.match(r"[A-Z]+\d+", part).group()
    for spk in sorted({p.name.split(".")[1] for p in (AMI/"words").glob(f"{part}.*.words.xml")}):
        words = read_words(AMI/"words"/f"{part}.{spk}.words.xml")
        for da in read_dialogue_acts(AMI/"dialogueActs"/f"{part}.{spk}.dialog-act.xml", words):
            rows.append({**da, "meeting": meeting, "part": part, "speaker": spk,
                         "label": int(da["da_id"] in positives)})

df = pd.DataFrame(rows)
df.to_parquet("utterances.parquet", index=False)

print(f"meetings: {df.meeting.nunique()}   utterances: {len(df)}   "
      f"action items: {df.label.sum()} ({df.label.mean():.2%})")
assert df.label.sum() == 381, "Expected 381 action items - the parse changed, check it."

meetings: 35   utterances: 115388   action items: 381 (0.33%)


## 3. Check the labels look right

Print a few positives next to the summary action they were linked to, so we can
see the labels make sense. Change the meeting name to spot-check any meeting.

In [34]:
def show_labels(part, limit=6):
    acts = action_sentences(AMI/"abstractive"/f"{part}.abssumm.xml")
    text = {nid(s): (s.text or "").strip()
            for sec in ET.parse(AMI/"abstractive"/f"{part}.abssumm.xml").getroot()
            if sec.tag == "actions" for s in sec if s.tag == "sentence"}
    words = {spk: read_words(AMI/"words"/f"{part}.{spk}.words.xml") for spk in "ABCDE"}
    def da_text(da_id):
        spk = da_id.split(".")[1]
        for d in ET.parse(AMI/"dialogueActs"/f"{part}.{spk}.dialog-act.xml").getroot():
            if d.tag == "dact" and nid(d) == da_id:
                for ch in d:
                    if ch.tag == NITE+"child" and "words" in ch.get("href",""):
                        return join(words[spk].get(w,"") for w in word_range(ch.get("href")))
        return "?"
    n = 0
    for link in ET.parse(AMI/"extractive"/f"{part}.summlink.xml").getroot():
        if link.tag != "summlink": continue
        da = ab = None
        for ptr in link:
            h = ptr.get("href","")
            if "dialog-act" in h: da = (ids(h) or [None])[0]
            elif "abssumm" in h:  ab = (ids(h) or [None])[0]
        if ab in acts:
            print("SAID   :", da_text(da)[:95])
            print("SUMMARY:", text[ab], "\n")
            n += 1
            if n >= limit: break

show_labels("ES2002b")

SAID   : um for uh our Industrial Designer, you're gonna be thinking about the components concept.
SUMMARY: The industrial designer will work on the components concept. 

SAID   : Um User Interface Designer gonna be thinking about our user interface,
SUMMARY: The user interface designer will work on the user interface. 

SAID   : and marketing you're gonna be thinking about trend watching.
SUMMARY: The marketing expert will work on trend watching 



## 4. Split into train / dev / test

We split by meeting, not by single utterance. If the same meeting showed up in
two splits, the model could cheat by memorizing its speakers and topics. These
five test and five dev meetings are the standard AMI choice.

In [35]:
TEST = {"ES2004","ES2014","IS1009","TS3003","TS3007"}
DEV  = {"ES2003","ES2011","IS1008","TS3004","TS3006"}
df["split"] = df.meeting.map(lambda m: "test" if m in TEST else "dev" if m in DEV else "train")
assert (df.groupby("meeting").split.nunique() == 1).all(), "a meeting leaked into two splits"

Path("splits").mkdir(exist_ok=True)
for s in ["train","dev","test"]:
    d = df[df.split == s]
    d.to_parquet(f"splits/{s}.parquet", index=False)
    # also save as jsonl (text + label) so the RoBERTa / LLaMA work loads the same rows
    d[["text","label","meeting","speaker"]].to_json(f"splits/{s}.jsonl", orient="records", lines=True)
    print(f"{s:5s}: {d.meeting.nunique():2d} meetings, {len(d):6d} utterances, {d.label.sum():3d} action items")

train: 25 meetings,  81080 utterances, 255 action items
dev  :  5 meetings,  17401 utterances,  73 action items
test :  5 meetings,  16907 utterances,  53 action items


## 5. Baseline: TF-IDF + Logistic Regression

Only about 0.3% of utterances are action items, so plain accuracy is useless (a
model that says "no" every time scores 99%). We report precision, recall, and F1
for the action-item class, balance the classes, and pick the cutoff on dev before
scoring test. The RoBERTa and LLaMA work should be scored the same way.

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

tr, dv, te = (df[df.split == s].assign(text=lambda x: x.text.fillna("")) for s in ["train","dev","test"])

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
).fit(tr.text, tr.label)

dev_prob = model.predict_proba(dv.text)[:,1]
cutoff = max(np.linspace(.05,.95,19), key=lambda t: f1_score(dv.label, dev_prob>=t, zero_division=0))

for name, d in [("dev", dv), ("test", te)]:
    pred = model.predict_proba(d.text)[:,1] >= cutoff
    print(f"\n[{name}] cutoff = {cutoff:.2f}")
    print(classification_report(d.label, pred, digits=3, zero_division=0))


[dev] cutoff = 0.90
              precision    recall  f1-score   support

           0      0.997     0.998     0.998     17328
           1      0.395     0.233     0.293        73

    accuracy                          0.995     17401
   macro avg      0.696     0.616     0.645     17401
weighted avg      0.994     0.995     0.995     17401


[test] cutoff = 0.90
              precision    recall  f1-score   support

           0      0.998     0.999     0.998     16854
           1      0.429     0.340     0.379        53

    accuracy                          0.997     16907
   macro avg      0.713     0.669     0.689     16907
weighted avg      0.996     0.997     0.996     16907



## 6. What kinds of utterances are action items (for the report)

Which dialogue-act types the action items fall into. Useful for the write-up.

In [37]:
(df.groupby("da_type").agg(total=("label","size"), action_items=("label","sum"))
   .query("action_items > 0").sort_values("action_items", ascending=False))

,total,action_items
da_type,,
ami_da_4,32726,257
ami_da_6,9176,69
ami_da_7,1493,30
ami_da_5,4128,10
ami_da_9,21069,6
ami_da_12,2302,4
ami_da_11,2236,2
ami_da_14,2185,1
ami_da_2,8251,1
